In [0]:
"""
id: python_3
template: python
templateVersion: 1.0.0
name: GitHub_API_Customers
position:
  x: 300
  y: 232.5
description:
  text: Load customer data from a CSV file if no input data is provided; otherwise, use the given data.
  hash: 56c26ad6
previewCodeHash: 6c840e93c531efcf
previewMode: "1000"
config:
  code: |+
    if inputs.get("data"):
        result = inputs["data"][0]
    else:
        import requests
        import pandas as pd
        from io import StringIO
        url="https://raw.githubusercontent.com/anshlambagit/Databricks_Lakeflow_Designer/refs/heads/main/customers.csv"

        response =requests.get(url)
        csv_data =response.content.decode('utf-8')
        df = pd.read_csv(StringIO(csv_data))

        result = spark.createDataFrame(df)
        
input: []
"""

# generated from the system
from typing import Dict, Any
from pyspark.sql import DataFrame

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    data = inputs.get("data", [] if True else None)
    result = data[0] if data else spark.createDataFrame([], "col: string")

    if inputs.get("data"):
        result = inputs["data"][0]
    else:
        import requests
        import pandas as pd
        from io import StringIO
        url="https://raw.githubusercontent.com/anshlambagit/Databricks_Lakeflow_Designer/refs/heads/main/customers.csv"

        response =requests.get(url)
        csv_data =response.content.decode('utf-8')
        df = pd.read_csv(StringIO(csv_data))

        result = spark.createDataFrame(df)

    return {"result": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {}
inputs = {}
out = run(config, inputs, spark)
ctx["python_3.result"] = out["result"]

In [0]:
"""
id: python_4
template: python
templateVersion: 1.0.0
name: API_Shipments
position:
  x: 600
  y: 310
description:
  text: Load data from an external CSV file if no input data is provided.
  hash: 100a60c6
previewCodeHash: 382df191413460f1
previewMode: "1000"
config:
  code: "if inputs.get(\"data\"):

    \    result = inputs[\"data\"][0]

    else:

    \    import requests

    \    import pandas as pd

    \    from io import StringIO

    \    url=\"https://raw.githubusercontent.com/anshlambagit/Databricks_Lakeflow_Designer/refs/heads/main/shipments.csv\"


    \    response =requests.get(url)

    \    csv_data =response.content.decode('utf-8')

    \    df = pd.read_csv(StringIO(csv_data))


    \    result = spark.createDataFrame(df)

    \    "
input: []
"""

# generated from the system
from typing import Dict, Any
from pyspark.sql import DataFrame

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    data = inputs.get("data", [] if True else None)
    result = data[0] if data else spark.createDataFrame([], "col: string")

    if inputs.get("data"):
        result = inputs["data"][0]
    else:
        import requests
        import pandas as pd
        from io import StringIO
        url="https://raw.githubusercontent.com/anshlambagit/Databricks_Lakeflow_Designer/refs/heads/main/shipments.csv"

        response =requests.get(url)
        csv_data =response.content.decode('utf-8')
        df = pd.read_csv(StringIO(csv_data))

        result = spark.createDataFrame(df)

    return {"result": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {}
inputs = {}
out = run(config, inputs, spark)
ctx["python_4.result"] = out["result"]

In [0]:
"""
id: source_0
template: source
templateVersion: 2.0.0
name: orders
position:
  x: 1.259921005475988
  y: 2.51984201095199
description:
  text: Load all data from the orders table.
  hash: cf94c49a
previewCodeHash: 4eecbde2cc17eb57
previewMode: "1000"
config:
  table_source:
    tableName: designer.raw.orders
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "designer.raw.orders"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["source_0.data"] = out["data"]

In [0]:
"""
id: source_1
template: source
templateVersion: 2.0.0
name: order_items
position:
  x: 0
  y: 155
description:
  text: Load all data from the order_items table in the raw designer database.
  hash: a54eb3f7
previewCodeHash: adb70a581bf8c732
previewMode: "1000"
config:
  table_source:
    tableName: designer.raw.order_items
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "designer.raw.order_items"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["source_1.data"] = out["data"]

In [0]:
"""
id: source_10
template: source
templateVersion: 2.0.0
name: Reviews
position:
  x: 0.870550563296149
  y: 478.907809146093
description:
  text: Load all data from the reviews table.
  hash: 29dc942a
previewCodeHash: 98ac819b5689474d
previewMode: "1000"
config:
  table_source:
    tableName: designer.raw.reviews
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "designer.raw.reviews"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["source_10.data"] = out["data"]

In [0]:
"""
id: join_2
template: join
templateVersion: 1.0.0
name: OrdersJoinOrdersItems
position:
  x: 300
  y: 77.5
description:
  text: Perform a left join on order_id and keep specified columns from both tables.
  hash: 9457e8db
previewCodeHash: d66029cd6329179e
previewMode: "1000"
config:
  join_type: left
  join_conditions: left.order_id = right.order_id
  expressions:
    - "`left`.order_id"
    - "`left`.customer_id"
    - "`left`.order_date"
    - "`left`.order_status"
    - "`right`.order_item_id"
    - "`right`.product_id"
    - "`right`.quantity"
    - "`right`.unit_price"
    - "`right`.discount_pct"
    - "`right`.line_total"
input:
  - node: source_0
    input_port: left
    output_port: data
  - node: source_1
    input_port: right
    output_port: data
"""

# generated from the system
from typing import Dict, Any, List
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_type = config.get("join_type", "inner").replace(" ", "_")
    join_condition = config.get("join_conditions", "")
    expressions: List[str] = config.get("expressions", [])

    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    if not join_condition:
        result = df_left.join(df_right, how=join_type)
    else:
        result = df_left.join(df_right, F.expr(join_condition), how=join_type)

    if expressions:
        result = result.selectExpr(*expressions)

    return {"joined_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "join_conditions": "left.order_id = right.order_id",
    "expressions": [
        "`left`.order_id",
        "`left`.customer_id",
        "`left`.order_date",
        "`left`.order_status",
        "`right`.order_item_id",
        "`right`.product_id",
        "`right`.quantity",
        "`right`.unit_price",
        "`right`.discount_pct",
        "`right`.line_total"
    ]
}
inputs = {
    "left": ctx["source_0.data"],
    "right": ctx["source_1.data"]
}
out = run(config, inputs, spark)
ctx["join_2.joined_data"] = out["joined_data"]

In [0]:
"""
id: ai_function_11
template: ai_function
templateVersion: 3.0.0
name: Sentiments_Analysis
position:
  x: 300.87055056329615
  y: 478.907809146093
description:
  text: Add sentiment analysis of review body to the data while keeping all original columns.
  hash: 9bdb680c
previewCodeHash: 4fe46069c2764989
previewMode: "1000"
config:
  expressions:
    - ai_analyze_sentiment(review_body) `sentiment`
  keep_all_columns: true
input:
  - node: source_10
    input_port: data
    output_port: data
"""

# generated from the system
from typing import Dict, Any, List
import hashlib

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]

    # Table-valued AI functions (ai_forecast, ...) live in the FROM
    # clause and have no SELECT-list shape, so we run them via
    # spark.sql and bind the upstream DataFrame as a session-scoped
    # temp view substituted in for the
    # __lakebuilder_ai_function_input__ placeholder identifier
    # (which is a real SQL identifier so the persisted statement
    # round-trips through the SQL parser when the cell reloads).
    #
    # spark.sql returns a lazy DataFrame: the view name is resolved
    # by Spark at action time (preview limit/collect), not when
    # spark.sql is called. We therefore deliberately do NOT drop
    # the temp view here — dropping it would leave the lazy plan
    # pointing at a missing relation and trigger TABLE_OR_VIEW_NOT
    # _FOUND when the downstream action fires.
    #
    # The view name is derived deterministically from (tvf_sql,
    # id(df)) so reruns of the same cell reuse the same name and
    # createOrReplaceTempView keeps the session catalog bounded at
    # one entry per (cell × upstream) instead of growing one entry
    # per run. id(df) is included to keep two cells that happen to
    # have identical tvf_sql but distinct upstream DataFrames from
    # clobbering each other's bindings.
    tvf_sql: str = config.get("tvf_sql") or ""
    if tvf_sql:
        key = f"{tvf_sql}\x00{id(df)}".encode("utf-8")
        view_name = f"lakebuilder_ai_fn_{hashlib.sha256(key).hexdigest()[:12]}"
        df.createOrReplaceTempView(view_name)
        sql = tvf_sql.replace("__lakebuilder_ai_function_input__", view_name)
        return {"ai_data": spark.sql(sql)}

    expressions: List[str] = config.get("expressions", [])
    if not expressions:
        return {"ai_data": df}

    keep_all_columns: bool = config.get("keep_all_columns", True)
    if keep_all_columns:
        return {"ai_data": df.selectExpr(*expressions, "*")}
    return {"ai_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "ai_analyze_sentiment(review_body) `sentiment`"
    ],
    "keep_all_columns": True
}
inputs = {
    "data": ctx["source_10.data"]
}
out = run(config, inputs, spark)
ctx["ai_function_11.ai_data"] = out["ai_data"]

In [0]:
"""
id: join_5
template: join
templateVersion: 1.0.0
name: Orders_OrdersItemsJoinAPICustomers
position:
  x: 600
  y: 155
description:
  text: Combine customer and order details by matching customer IDs, keeping all orders and their related customer info.
  hash: 76c404ec
previewCodeHash: 8f53aad8c13b8d24
previewMode: "1000"
config:
  join_type: left
  join_conditions: left.customer_id = right.customer_id
  expressions:
    - "`right`.customer_name"
    - "`right`.email"
    - "`right`.phone"
    - "`right`.city"
    - "`right`.state"
    - "`right`.country"
    - "`right`.created_date"
    - "`left`.order_id"
    - "`left`.order_date"
    - "`left`.order_status"
    - "`left`.order_item_id"
    - "`left`.product_id"
    - "`left`.quantity"
    - "`left`.unit_price"
    - "`left`.discount_pct"
    - "`left`.line_total"
    - "`left`.customer_id"
input:
  - node: join_2
    input_port: left
    output_port: joined_data
  - node: python_3
    input_port: right
    output_port: result
"""

# generated from the system
from typing import Dict, Any, List
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_type = config.get("join_type", "inner").replace(" ", "_")
    join_condition = config.get("join_conditions", "")
    expressions: List[str] = config.get("expressions", [])

    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    if not join_condition:
        result = df_left.join(df_right, how=join_type)
    else:
        result = df_left.join(df_right, F.expr(join_condition), how=join_type)

    if expressions:
        result = result.selectExpr(*expressions)

    return {"joined_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "join_conditions": "left.customer_id = right.customer_id",
    "expressions": [
        "`right`.customer_name",
        "`right`.email",
        "`right`.phone",
        "`right`.city",
        "`right`.state",
        "`right`.country",
        "`right`.created_date",
        "`left`.order_id",
        "`left`.order_date",
        "`left`.order_status",
        "`left`.order_item_id",
        "`left`.product_id",
        "`left`.quantity",
        "`left`.unit_price",
        "`left`.discount_pct",
        "`left`.line_total",
        "`left`.customer_id"
    ]
}
inputs = {
    "left": ctx["join_2.joined_data"],
    "right": ctx["python_3.result"]
}
out = run(config, inputs, spark)
ctx["join_5.joined_data"] = out["joined_data"]

In [0]:
"""
id: ai_function_13
template: ai_function
templateVersion: 3.0.0
name: Translate_Reviews_English
position:
  x: 578.5626923788332
  y: 709.6037084195657
description:
  text: Translate review_body to Hindi and keep all original columns.
  hash: 7c7c8f0d
previewCodeHash: e3c7788088d8d9a0
previewMode: "1000"
config:
  expressions:
    - ai_translate(review_body, 'Hindi') `review_body_Hindi`
  keep_all_columns: true
input:
  - node: ai_function_11
    input_port: data
    output_port: ai_data
"""

# generated from the system
from typing import Dict, Any, List
import hashlib

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]

    # Table-valued AI functions (ai_forecast, ...) live in the FROM
    # clause and have no SELECT-list shape, so we run them via
    # spark.sql and bind the upstream DataFrame as a session-scoped
    # temp view substituted in for the
    # __lakebuilder_ai_function_input__ placeholder identifier
    # (which is a real SQL identifier so the persisted statement
    # round-trips through the SQL parser when the cell reloads).
    #
    # spark.sql returns a lazy DataFrame: the view name is resolved
    # by Spark at action time (preview limit/collect), not when
    # spark.sql is called. We therefore deliberately do NOT drop
    # the temp view here — dropping it would leave the lazy plan
    # pointing at a missing relation and trigger TABLE_OR_VIEW_NOT
    # _FOUND when the downstream action fires.
    #
    # The view name is derived deterministically from (tvf_sql,
    # id(df)) so reruns of the same cell reuse the same name and
    # createOrReplaceTempView keeps the session catalog bounded at
    # one entry per (cell × upstream) instead of growing one entry
    # per run. id(df) is included to keep two cells that happen to
    # have identical tvf_sql but distinct upstream DataFrames from
    # clobbering each other's bindings.
    tvf_sql: str = config.get("tvf_sql") or ""
    if tvf_sql:
        key = f"{tvf_sql}\x00{id(df)}".encode("utf-8")
        view_name = f"lakebuilder_ai_fn_{hashlib.sha256(key).hexdigest()[:12]}"
        df.createOrReplaceTempView(view_name)
        sql = tvf_sql.replace("__lakebuilder_ai_function_input__", view_name)
        return {"ai_data": spark.sql(sql)}

    expressions: List[str] = config.get("expressions", [])
    if not expressions:
        return {"ai_data": df}

    keep_all_columns: bool = config.get("keep_all_columns", True)
    if keep_all_columns:
        return {"ai_data": df.selectExpr(*expressions, "*")}
    return {"ai_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "ai_translate(review_body, 'Hindi') `review_body_Hindi`"
    ],
    "keep_all_columns": True
}
inputs = {
    "data": ctx["ai_function_11.ai_data"]
}
out = run(config, inputs, spark)
ctx["ai_function_13.ai_data"] = out["ai_data"]

In [0]:
"""
id: join_6
template: join
templateVersion: 1.0.0
name: BigTable
position:
  x: 900
  y: 232.5
description:
  text: Left join on order_id to combine order and shipment details, keeping selected columns from both sides.
  hash: dab8c68c
previewCodeHash: 4b9a94ce150d6c22
previewMode: "1000"
config:
  join_type: left
  join_conditions: left.order_id = right.order_id
  expressions:
    - "`left`.customer_name"
    - "`left`.email"
    - "`left`.phone"
    - "`left`.city"
    - "`left`.state"
    - "`left`.country"
    - "`left`.created_date"
    - "`left`.order_id"
    - "`left`.order_date"
    - "`left`.order_status"
    - "`left`.order_item_id"
    - "`left`.product_id"
    - "`left`.quantity"
    - "`left`.unit_price"
    - "`left`.discount_pct"
    - "`left`.line_total"
    - "`left`.customer_id"
    - "`right`.shipment_id"
    - "`right`.ship_date"
    - "`right`.ship_mode"
    - "`right`.shipping_cost"
input:
  - node: join_5
    input_port: left
    output_port: joined_data
  - node: python_4
    input_port: right
    output_port: result
"""

# generated from the system
from typing import Dict, Any, List
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_type = config.get("join_type", "inner").replace(" ", "_")
    join_condition = config.get("join_conditions", "")
    expressions: List[str] = config.get("expressions", [])

    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    if not join_condition:
        result = df_left.join(df_right, how=join_type)
    else:
        result = df_left.join(df_right, F.expr(join_condition), how=join_type)

    if expressions:
        result = result.selectExpr(*expressions)

    return {"joined_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "join_conditions": "left.order_id = right.order_id",
    "expressions": [
        "`left`.customer_name",
        "`left`.email",
        "`left`.phone",
        "`left`.city",
        "`left`.state",
        "`left`.country",
        "`left`.created_date",
        "`left`.order_id",
        "`left`.order_date",
        "`left`.order_status",
        "`left`.order_item_id",
        "`left`.product_id",
        "`left`.quantity",
        "`left`.unit_price",
        "`left`.discount_pct",
        "`left`.line_total",
        "`left`.customer_id",
        "`right`.shipment_id",
        "`right`.ship_date",
        "`right`.ship_mode",
        "`right`.shipping_cost"
    ]
}
inputs = {
    "left": ctx["join_5.joined_data"],
    "right": ctx["python_4.result"]
}
out = run(config, inputs, spark)
ctx["join_6.joined_data"] = out["joined_data"]

In [0]:
"""
id: output_12
template: output
templateVersion: 1.0.0
name: sentiment
position:
  x: 860.8705505632962
  y: 478.907809146093
description:
  text: Save data by overwriting the specified table.
  hash: 8b51ecd7
previewMode: "1000"
config:
  catalog: designer
  schema: enrich
  table_name: sentiments
input:
  - node: ai_function_13
    input_port: data
    output_port: ai_data
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    parts = [p for p in [catalog, schema, table_name] if p]
    full_name = ".".join(parts)

    df.write.mode("overwrite").saveAsTable(full_name)

    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "catalog": "designer",
    "schema": "enrich",
    "table_name": "sentiments"
}
inputs = {
    "data": ctx["ai_function_13.ai_data"]
}
out = run(config, inputs, spark)

In [0]:
"""
id: output_14
template: output
templateVersion: 1.0.0
name: ReviewsInHindi
position:
  x: 838.5626923788332
  y: 709.6037084195657
description:
  text: Save data to the specified table, replacing the existing content.
  hash: 10b06de2
previewMode: "1000"
config:
  catalog: designer
  schema: enrich
  table_name: ReviewHindi
input:
  - node: ai_function_13
    input_port: data
    output_port: ai_data
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    parts = [p for p in [catalog, schema, table_name] if p]
    full_name = ".".join(parts)

    df.write.mode("overwrite").saveAsTable(full_name)

    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "catalog": "designer",
    "schema": "enrich",
    "table_name": "ReviewHindi"
}
inputs = {
    "data": ctx["ai_function_13.ai_data"]
}
out = run(config, inputs, spark)

In [0]:
"""
id: aggregate_7
template: aggregate
templateVersion: 1.0.0
name: OrdersByCity
position:
  x: 1200
  y: 232.5
description:
  text: Group data by city and count orders, also calculate total amounts.
  hash: f43a43fd
previewCodeHash: 9052c3aebe57a550
previewMode: "1000"
config:
  group_bys:
    - expr: city
      type: column
  aggregations:
    - columnExpr:
        expr: order_id
        type: column
      fn: COUNT
      alias: TotalOrders
    - columnExpr:
        expr: Round(SUM(unit_price), 2)
        type: expr
      fn: "-"
      alias: TotalAmounts
input:
  - node: join_6
    input_port: data
    output_port: joined_data
"""

# generated from the system
import math
from typing import Dict, Any
import pyspark.sql.functions as F

DEFAULT_PERCENTILE = 0.5

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    group_bys = config.get("group_bys", [])
    aggregations = config.get("aggregations", [])

    group_by_set = set(e for gb in group_bys if (e := gb.get("expr", "")))

    agg_exprs = []
    for agg_def in aggregations:
        col_expr = agg_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        fn = agg_def.get("fn", "-")
        alias = agg_def.get("alias")

        if (fn == "-" or fn == "_") and not alias and raw_expr in group_by_set:
            continue

        fn_map = {
            "SUM": F.sum,
            "AVG": F.avg,
            "COUNT": F.count,
            "MIN": F.min,
            "MAX": F.max,
            "MEAN": F.mean,
            "MEDIAN": F.median,
            "STDDEV": F.stddev,
            "VARIANCE": F.variance,
        }

        agg_fn = fn_map.get(fn)
        if agg_fn:
            col = agg_fn(raw_expr)
        elif fn == "-" or fn == "_":
            col = F.expr(raw_expr) if col_expr.get("type") == "expr" else F.col(raw_expr)
        elif fn == "PERCENTILE":
            raw_pct = agg_def.get("percentage")
            if (
                isinstance(raw_pct, (int, float))
                and not isinstance(raw_pct, bool)
                and math.isfinite(raw_pct)
            ):
                pct = max(0.0, min(1.0, float(raw_pct)))
            else:
                pct = DEFAULT_PERCENTILE
            col = F.expr(f"PERCENTILE({raw_expr}, {pct})")
        else:
            col = F.expr(f"{fn}({raw_expr})")

        if alias:
            col = col.alias(alias)

        agg_exprs.append(col)

    group_cols = [
        gb.get("expr", "") for gb in group_bys if gb.get("expr", "")
    ]

    if not agg_exprs:
        if group_cols:
            result = df.select(*group_cols).distinct()
            return {"aggregated_data": result}
        return {"aggregated_data": df}

    if group_cols:
        result = df.groupBy(*group_cols).agg(*agg_exprs)
    else:
        result = df.agg(*agg_exprs)

    return {"aggregated_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "group_bys": [
        {
            "expr": "city",
            "type": "column"
        }
    ],
    "aggregations": [
        {
            "columnExpr": {
                "expr": "order_id",
                "type": "column"
            },
            "fn": "COUNT",
            "alias": "TotalOrders",
            "withAsKeyword": None
        },
        {
            "columnExpr": {
                "expr": "Round(SUM(unit_price), 2)",
                "type": "expr"
            },
            "fn": "-",
            "alias": "TotalAmounts",
            "withAsKeyword": None
        }
    ]
}
inputs = {
    "data": ctx["join_6.joined_data"]
}
out = run(config, inputs, spark)
ctx["aggregate_7.aggregated_data"] = out["aggregated_data"]

In [0]:
"""
id: sort_8
template: sort
templateVersion: 1.0.0
name: Sorted
position:
  x: 1500
  y: 232.5
description:
  text: Sort data by TotalAmounts in descending order.
  hash: 968412b7
previewCodeHash: 0d501403056d9b40
previewMode: "1000"
config:
  sort_expressions:
    - columnExpr:
        expr: TotalAmounts
        type: column
      sortBy: DESC
input:
  - node: aggregate_7
    input_port: data
    output_port: aggregated_data
"""

# generated from the system
from typing import Dict, Any
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    sort_expressions = config.get("sort_expressions", [])

    if not sort_expressions:
        return {"sorted_data": df}

    order_cols = []
    for sort_def in sort_expressions:
        col_expr = sort_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        direction = sort_def.get("sortBy", "UNSET")

        col = F.col(raw_expr)
        if direction == "DESC":
            col = col.desc()
        elif direction == "ASC":
            col = col.asc()

        order_cols.append(col)

    return {"sorted_data": df.orderBy(*order_cols)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "sort_expressions": [
        {
            "columnExpr": {
                "expr": "TotalAmounts",
                "type": "column"
            },
            "sortBy": "DESC"
        }
    ]
}
inputs = {
    "data": ctx["aggregate_7.aggregated_data"]
}
out = run(config, inputs, spark)
ctx["sort_8.sorted_data"] = out["sorted_data"]

In [0]:
"""
id: output_9
template: output
templateVersion: 1.0.0
name: designer.enrich.Aggregated_Orders
position:
  x: 1800.8705505448813
  y: 231.6294494551188
description:
  text: Save data by replacing existing table in designer.enrich.Aggregated_Orders.
  hash: 9878b372
previewMode: "1000"
config:
  catalog: designer
  schema: enrich
  table_name: Aggregated_Orders
input:
  - node: sort_8
    input_port: data
    output_port: sorted_data
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    parts = [p for p in [catalog, schema, table_name] if p]
    full_name = ".".join(parts)

    df.write.mode("overwrite").saveAsTable(full_name)

    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "catalog": "designer",
    "schema": "enrich",
    "table_name": "Aggregated_Orders"
}
inputs = {
    "data": ctx["sort_8.sorted_data"]
}
out = run(config, inputs, spark)